# Notebook 14 — SGLang and Advanced Scheduling Patterns

SGLang matters because it treats serving performance as a scheduling problem, not only a model-loading problem. This notebook uses notebook-local simulations to study how cache awareness, batching policy, chunked prefill, and structured-output lanes change runtime behavior.

## What you will build

- a synthetic request mix with repeated prefixes, long prompts, and JSON-heavy traffic
- a FIFO baseline scheduler and two progressively smarter policies
- a chunked-prefill simulation for protecting short interactive requests
- a reserved-lane experiment for structured outputs and tool calls
- scheduler artifacts that summarize when SGLang-style policies are worth the extra complexity

## Reconnecting to earlier runtime notebooks

Notebooks 10 through 13 explained continuous batching, prefix caching, structured outputs, and multi-model routing. Here we combine those ideas into one runtime control surface: which requests are admitted together, which prefixes are reused, and which latency budgets get protected when workloads collide.

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

## Step 1: Add notebook helpers and an artifact directory

The notebook stays lightweight: plain pandas tables, small deterministic simulations, and JSON artifacts that could feed a later routing or observability notebook.

In [ ]:
random.seed(14)

ARTIFACT_DIR = ARTIFACT_ROOT / "14_sglang_and_advanced_scheduling_patterns"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def percentile(values, pct):
    values = [float(v) for v in values]
    if not values:
        return 0.0
    return float(np.percentile(values, pct))

def clipped(value, low, high):
    return max(low, min(high, value))

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2: Build a mixed request stream

The request mix includes repeated system prompts, long-context jobs, and JSON-constrained requests. That combination creates the same scheduler tension you see in real serving systems: throughput wants batching, but user-facing latency wants protection from long or awkward requests.

In [ ]:
profiles = [
    {"workload": "chat_turn", "base_prompt_tokens": 900, "base_output_tokens": 180, "shared_prefix_tokens": 520, "json_ratio": 0.15, "priority": "standard"},
    {"workload": "tool_json", "base_prompt_tokens": 760, "base_output_tokens": 120, "shared_prefix_tokens": 430, "json_ratio": 1.00, "priority": "high"},
    {"workload": "long_context", "base_prompt_tokens": 5200, "base_output_tokens": 220, "shared_prefix_tokens": 1800, "json_ratio": 0.05, "priority": "standard"},
    {"workload": "retrieval_followup", "base_prompt_tokens": 2500, "base_output_tokens": 140, "shared_prefix_tokens": 1350, "json_ratio": 0.30, "priority": "standard"},
    {"workload": "ops_triage", "base_prompt_tokens": 520, "base_output_tokens": 95, "shared_prefix_tokens": 260, "json_ratio": 0.72, "priority": "urgent"},
]

records = []
arrival_clock = 0.0
for index in range(60):
    profile = profiles[index % len(profiles)]
    rng = random.Random(f"sglang-{index}")
    burst_gap = rng.uniform(2, 13) if index % 9 else rng.uniform(18, 34)
    arrival_clock += burst_gap
    prompt_tokens = max(120, int(rng.gauss(profile["base_prompt_tokens"], profile["base_prompt_tokens"] * 0.18)))
    output_tokens = max(32, int(rng.gauss(profile["base_output_tokens"], profile["base_output_tokens"] * 0.20)))
    shared_prefix_tokens = min(prompt_tokens - 32, max(0, int(rng.gauss(profile["shared_prefix_tokens"], max(profile["shared_prefix_tokens"] * 0.10, 1)))))
    requires_json = rng.random() < profile["json_ratio"]
    records.append({
        "request_id": f"req_{index:03d}",
        "arrival_ms": round(arrival_clock, 2),
        "workload": profile["workload"],
        "prompt_tokens": prompt_tokens,
        "output_tokens": output_tokens,
        "shared_prefix_tokens": shared_prefix_tokens,
        "prefix_group": f'{profile["workload"]}_{index % 4}',
        "requires_json": requires_json,
        "priority": profile["priority"],
    })

requests_df = pd.DataFrame(records)
requests_df.head(10)

## Step 3: Inspect the scheduler pressure before choosing a policy

The scheduler cares about prompt length spread, repeated prefixes, and whether some requests need more deterministic decode behavior than others.

In [ ]:
mix_summary = (
    requests_df.groupby("workload")
    .agg(
        requests=("request_id", "count"),
        avg_prompt_tokens=("prompt_tokens", "mean"),
        avg_output_tokens=("output_tokens", "mean"),
        avg_shared_prefix_tokens=("shared_prefix_tokens", "mean"),
        json_share=("requires_json", "mean"),
    )
    .round(2)
    .reset_index()
)
mix_summary["prefix_share"] = (mix_summary["avg_shared_prefix_tokens"] / mix_summary["avg_prompt_tokens"]).round(3)
mix_summary

## Step 4: Define scheduler profiles

We will compare three policies: a FIFO baseline, a length-aware packer, and a cache-aware policy inspired by SGLang scheduling. The SGLang-style profile also gets slightly faster prefill and decode settings because smarter scheduling normally comes with runtime engineering, not only better ordering.

In [ ]:
scheduler_profiles = pd.DataFrame([
    {"policy": "fifo", "dispatch_wait_ms": 6, "max_batch_size": 4, "max_batch_tokens": 7600, "prefill_tps": 7600, "decode_step_ms": 12.6, "prefix_discount": 0.00},
    {"policy": "length_aware", "dispatch_wait_ms": 8, "max_batch_size": 5, "max_batch_tokens": 7600, "prefill_tps": 8100, "decode_step_ms": 11.9, "prefix_discount": 0.18},
    {"policy": "prefix_aware", "dispatch_wait_ms": 10, "max_batch_size": 6, "max_batch_tokens": 9000, "prefill_tps": 9000, "decode_step_ms": 10.8, "prefix_discount": 0.62},
])
scheduler_profiles

## Step 5: Implement a scheduler simulator

The simulator is intentionally approximate. It tracks queueing, prefill cost, decode cost, and prefix reuse. That is enough to compare policies without pretending a notebook should faithfully emulate a GPU runtime.

In [ ]:
def prompt_band(prompt_tokens):
    if prompt_tokens < 900:
        return "short"
    if prompt_tokens < 2600:
        return "medium"
    return "long"

def pick_batch(ready, config):
    pool = ready[:]
    if config["policy"] == "fifo":
        pool.sort(key=lambda item: (item["arrival_ms"], item["request_id"]))
    elif config["policy"] == "length_aware":
        pool.sort(key=lambda item: (item["prompt_tokens"], item["arrival_ms"]))
    else:
        anchor = min(pool, key=lambda item: item["arrival_ms"])
        pool.sort(key=lambda item: (-(item["prefix_group"] == anchor["prefix_group"]), -item["shared_prefix_tokens"], item["arrival_ms"]))

    batch = []
    token_budget = 0
    for item in pool:
        projected = token_budget + item["prompt_tokens"]
        if batch and (len(batch) >= config["max_batch_size"] or projected > config["max_batch_tokens"]):
            continue
        batch.append(item)
        token_budget = projected
        if len(batch) >= config["max_batch_size"]:
            break
    return batch

def run_scheduler(records, config):
    pending = sorted((dict(row) for row in records), key=lambda item: item["arrival_ms"])
    waiting = []
    outputs = []
    clock = 0.0
    batch_id = 0

    while pending or waiting:
        if not waiting and pending:
            clock = max(clock, pending[0]["arrival_ms"])

        window_end = clock + config["dispatch_wait_ms"]
        while pending and pending[0]["arrival_ms"] <= window_end:
            waiting.append(pending.pop(0))

        if not waiting:
            continue

        batch = pick_batch(waiting, config)
        if not batch:
            clock = max(clock, pending[0]["arrival_ms"]) if pending else clock
            continue

        dispatch_ms = max(clock, min(item["arrival_ms"] for item in batch))
        prefix_groups = Counter(item["prefix_group"] for item in batch)
        reusable_tokens = 0
        for group, count in prefix_groups.items():
            if count < 2:
                continue
            prefix_candidates = [item["shared_prefix_tokens"] for item in batch if item["prefix_group"] == group]
            reusable_tokens += min(prefix_candidates) * (count - 1)
        effective_prompt_tokens = max(
            max(item["prompt_tokens"] for item in batch),
            sum(item["prompt_tokens"] for item in batch) - reusable_tokens * config["prefix_discount"]
        )
        prefill_ms = 1000 * effective_prompt_tokens / (config["prefill_tps"] * (1 + 0.10 * (len(batch) - 1)))
        decode_steps = max(item["output_tokens"] for item in batch)
        decode_ms = decode_steps * (config["decode_step_ms"] + 0.45 * (len(batch) - 1))
        overhead_ms = 22 + 5 * len(batch)
        finish_ms = dispatch_ms + prefill_ms + decode_ms + overhead_ms

        for item in batch:
            waiting.remove(item)
            queue_ms = dispatch_ms - item["arrival_ms"]
            ttft_ms = queue_ms + prefill_ms + 14
            outputs.append({
                **item,
                "policy": config["policy"],
                "batch_id": batch_id,
                "batch_size": len(batch),
                "dispatch_ms": round(dispatch_ms, 2),
                "finish_ms": round(finish_ms, 2),
                "queue_ms": round(queue_ms, 2),
                "ttft_ms": round(ttft_ms, 2),
                "latency_ms": round(finish_ms - item["arrival_ms"], 2),
                "cache_reuse_tokens": round(reusable_tokens / max(len(batch), 1), 2),
                "prompt_band": prompt_band(item["prompt_tokens"]),
            })

        clock = finish_ms
        batch_id += 1

    return pd.DataFrame(outputs)

request_records = requests_df.to_dict("records")

## Step 6: Run the FIFO baseline

FIFO is simple and predictable, but it does not understand prompt length spread or prefix locality. That usually means poor queue behavior once long-context requests arrive.

In [ ]:
fifo_config = scheduler_profiles.set_index("policy").loc["fifo"].to_dict()
fifo_df = run_scheduler(request_records, fifo_config)
fifo_df[["request_id", "workload", "batch_id", "batch_size", "queue_ms", "ttft_ms", "latency_ms"]].head(12)

## Step 7: Add a length-aware packer

Length-aware batching is a common first upgrade. It reduces padding-style waste and keeps similarly sized prompts together, but it still misses opportunities from repeated prefixes.

In [ ]:
length_config = scheduler_profiles.set_index("policy").loc["length_aware"].to_dict()
length_df = run_scheduler(request_records, length_config)
length_df[["request_id", "workload", "batch_id", "batch_size", "queue_ms", "ttft_ms", "latency_ms"]].head(12)

## Step 8: Add a prefix-aware continuous-batching policy

This is the SGLang-style step. The policy waits slightly longer, forms larger batches, and tries to keep repeated prefixes together so prefill work can be reused instead of repeated.

In [ ]:
prefix_config = scheduler_profiles.set_index("policy").loc["prefix_aware"].to_dict()
prefix_df = run_scheduler(request_records, prefix_config)
prefix_df[["request_id", "workload", "batch_id", "batch_size", "queue_ms", "ttft_ms", "latency_ms", "cache_reuse_tokens"]].head(12)

## Step 9: Compare policy summaries

The useful question is not whether a scheduler is clever. The useful question is whether it improves throughput enough to justify any extra waiting, complexity, or fairness risk.

In [ ]:
def summarize_policy(frame):
    busy_window_s = max((frame["finish_ms"].max() - frame["arrival_ms"].min()) / 1000, 0.001)
    return {
        "policy": frame["policy"].iloc[0],
        "requests": int(len(frame)),
        "avg_batch_size": round(frame["batch_size"].mean(), 2),
        "mean_queue_ms": round(frame["queue_ms"].mean(), 2),
        "p95_ttft_ms": round(percentile(frame["ttft_ms"], 95), 2),
        "p95_latency_ms": round(percentile(frame["latency_ms"], 95), 2),
        "output_tps": round(frame["output_tokens"].sum() / busy_window_s, 2),
        "avg_cache_reuse_tokens": round(frame["cache_reuse_tokens"].mean(), 2),
    }

policy_summary = pd.DataFrame([
    summarize_policy(fifo_df),
    summarize_policy(length_df),
    summarize_policy(prefix_df),
]).sort_values("output_tps", ascending=False)
policy_summary

## Step 10: Visualize the latency and throughput trade-off

Good runtime reviews make trade-offs visible. A policy can raise throughput and still be the wrong choice if interactive TTFT becomes unacceptable.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
policy_summary.plot(x="policy", y=["p95_ttft_ms", "p95_latency_ms"], kind="bar", rot=0, ax=axes[0], color=["#4c78a8", "#f58518"])
policy_summary.plot(x="policy", y="output_tps", kind="bar", rot=0, ax=axes[1], color="#54a24b", legend=False)
axes[0].set_title("Tail latency by scheduler")
axes[0].set_ylabel("milliseconds")
axes[1].set_title("Approximate output throughput")
axes[1].set_ylabel("tokens per second")
plt.tight_layout()

## Step 11: Measure fairness, not only averages

Smarter batching can accidentally starve long or awkward requests. We will inspect prompt bands separately so the scheduler is judged by the user experience it creates, not only its aggregate efficiency.

In [ ]:
combined = pd.concat([fifo_df, length_df, prefix_df], ignore_index=True)
fairness_df = (
    combined.groupby(["policy", "prompt_band"])
    .agg(
        requests=("request_id", "count"),
        mean_queue_ms=("queue_ms", "mean"),
        p95_latency_ms=("latency_ms", lambda values: percentile(values, 95)),
    )
    .round(2)
    .reset_index()
)
fairness_df

## Step 12: Simulate chunked prefill under mixed long and short traffic

Chunked prefill exists because one very long prompt can otherwise block many short requests from getting their first token. The simulation below compares a monolithic prefill pass with a scheduler that can yield between chunks.

In [ ]:
chunk_scenario = [
    {"request_id": "long_0", "arrival_ms": 0.0, "prompt_tokens": 12000, "output_tokens": 240},
    {"request_id": "short_1", "arrival_ms": 40.0, "prompt_tokens": 420, "output_tokens": 96},
    {"request_id": "short_2", "arrival_ms": 66.0, "prompt_tokens": 520, "output_tokens": 88},
    {"request_id": "short_3", "arrival_ms": 92.0, "prompt_tokens": 610, "output_tokens": 104},
]

def simulate_prefill_strategy(records, chunk_tokens=None, prefill_tps=9000, decode_step_ms=11.5):
    rows = []
    long_request = dict(records[0])
    short_requests = [dict(item) for item in records[1:]]
    long_prefill_remaining = long_request["prompt_tokens"]
    clock = long_request["arrival_ms"]
    long_first_token_ms = None
    served_short = set()

    if chunk_tokens is None:
        prefill_ms = 1000 * long_request["prompt_tokens"] / prefill_tps
        long_first_token_ms = clock + prefill_ms + 14
        long_finish_ms = long_first_token_ms + long_request["output_tokens"] * decode_step_ms
        rows.append({"strategy": "monolithic_prefill", "request_id": long_request["request_id"], "ttft_ms": round(long_first_token_ms, 2), "finish_ms": round(long_finish_ms, 2)})
        for item in short_requests:
            start_ms = max(item["arrival_ms"], long_first_token_ms)
            ttft_ms = start_ms + (1000 * item["prompt_tokens"] / prefill_tps) + 14
            finish_ms = ttft_ms + item["output_tokens"] * (decode_step_ms - 1.6)
            rows.append({"strategy": "monolithic_prefill", "request_id": item["request_id"], "ttft_ms": round(ttft_ms - item["arrival_ms"], 2), "finish_ms": round(finish_ms - item["arrival_ms"], 2)})
        return pd.DataFrame(rows)

    while long_prefill_remaining > 0:
        processed = min(chunk_tokens, long_prefill_remaining)
        clock += 1000 * processed / prefill_tps
        long_prefill_remaining -= processed
        ready_short = [item for item in short_requests if item["arrival_ms"] <= clock and item["request_id"] not in served_short]
        for item in ready_short:
            ttft_ms = clock - item["arrival_ms"] + (1000 * item["prompt_tokens"] / prefill_tps) + 14
            finish_ms = ttft_ms + item["output_tokens"] * (decode_step_ms - 1.6)
            rows.append({"strategy": "chunked_prefill", "request_id": item["request_id"], "ttft_ms": round(ttft_ms, 2), "finish_ms": round(finish_ms, 2)})
            served_short.add(item["request_id"])
            clock += 18
        if long_first_token_ms is None and long_prefill_remaining <= 0:
            long_first_token_ms = clock + 14

    long_finish_ms = long_first_token_ms + long_request["output_tokens"] * decode_step_ms
    rows.append({"strategy": "chunked_prefill", "request_id": long_request["request_id"], "ttft_ms": round(long_first_token_ms, 2), "finish_ms": round(long_finish_ms, 2)})
    for item in short_requests:
        if item["request_id"] in served_short:
            continue
        ttft_ms = clock - item["arrival_ms"] + (1000 * item["prompt_tokens"] / prefill_tps) + 14
        finish_ms = ttft_ms + item["output_tokens"] * (decode_step_ms - 1.6)
        rows.append({"strategy": "chunked_prefill", "request_id": item["request_id"], "ttft_ms": round(ttft_ms, 2), "finish_ms": round(finish_ms, 2)})

    return pd.DataFrame(rows)

prefill_compare = pd.concat([
    simulate_prefill_strategy(chunk_scenario, chunk_tokens=None),
    simulate_prefill_strategy(chunk_scenario, chunk_tokens=1500),
], ignore_index=True)
prefill_compare.sort_values(["strategy", "request_id"])

## Step 13: Reserve a decode lane for JSON-heavy traffic

Structured-output workloads often care more about predictable tail latency than raw throughput. Reserving a lane can protect those requests, but it also removes capacity from free-form generations.

In [ ]:
def simulate_json_lanes(records, reserve_json_lane=False):
    general_clock = 0.0
    json_clock = 0.0
    rows = []
    ordered = sorted(records, key=lambda item: item["arrival_ms"])
    for item in ordered:
        service_ms = 180 + 0.028 * item["prompt_tokens"] + 1.05 * item["output_tokens"]
        if reserve_json_lane and item["requires_json"]:
            start_ms = max(item["arrival_ms"], json_clock)
            finish_ms = start_ms + service_ms
            json_clock = finish_ms
            lane = "json_lane"
        else:
            slowdown = 1.08 if reserve_json_lane else 1.0
            start_ms = max(item["arrival_ms"], general_clock)
            finish_ms = start_ms + service_ms * slowdown
            general_clock = finish_ms
            lane = "general_lane"
        rows.append({
            "scenario": "reserved_lane" if reserve_json_lane else "shared_lane",
            "request_id": item["request_id"],
            "requires_json": item["requires_json"],
            "lane": lane,
            "latency_ms": round(finish_ms - item["arrival_ms"], 2),
        })
    return pd.DataFrame(rows)

lane_compare = pd.concat([
    simulate_json_lanes(request_records, reserve_json_lane=False),
    simulate_json_lanes(request_records, reserve_json_lane=True),
], ignore_index=True)

lane_summary = (
    lane_compare.groupby(["scenario", "requires_json"])
    .agg(
        requests=("request_id", "count"),
        mean_latency_ms=("latency_ms", "mean"),
        p95_latency_ms=("latency_ms", lambda values: percentile(values, 95)),
    )
    .round(2)
    .reset_index()
)
lane_summary

## Step 14: Export scheduler artifacts and recommendations

At this point the notebook has enough evidence to produce a reusable artifact: which policy wins, how much cache reuse it achieved, and what trade-offs showed up in fairness or JSON latency.

In [ ]:
recommendations = []
best_policy = policy_summary.sort_values(["output_tps", "p95_ttft_ms"], ascending=[False, True]).iloc[0]["policy"]
for row in policy_summary.to_dict("records"):
    tradeoff = "balanced"
    if row["p95_ttft_ms"] > policy_summary["p95_ttft_ms"].min() * 1.20:
        tradeoff = "throughput_heavy"
    if row["output_tps"] < policy_summary["output_tps"].max() * 0.92:
        tradeoff = "latency_favoring"
    recommendations.append({
        "policy": row["policy"],
        "tradeoff": tradeoff,
        "use_when": "prefix reuse is common" if row["avg_cache_reuse_tokens"] > 300 else "workloads are mixed but prefix locality is weak",
    })

artifact = {
    "notebook": "14_sglang_and_advanced_scheduling_patterns",
    "policy_summary": policy_summary.to_dict("records"),
    "fairness": fairness_df.to_dict("records"),
    "chunked_prefill": prefill_compare.to_dict("records"),
    "lane_summary": lane_summary.to_dict("records"),
    "best_policy": best_policy,
    "recommendations": recommendations,
}

artifact_path = ARTIFACT_DIR / "sglang_scheduler_summary.json"
write_json(artifact_path, artifact)
print("Best policy:", best_policy)
print("Wrote:", artifact_path.resolve())

## Recap

SGLang-style scheduling is valuable when repeated prefixes, long prompts, and structured workloads compete for the same runtime. The gains do not come from magic. They come from smarter queue formation, more reuse during prefill, and explicit protection for latency-sensitive traffic.

In [ ]:
summary_rows = len(policy_summary)
assert summary_rows == 3
assert policy_summary["output_tps"].max() >= policy_summary["output_tps"].min()
assert lane_summary["p95_latency_ms"].max() > 0
print("Notebook 14 sanity checks passed.")